In [1]:
from rsl_rl.env import VecEnv
from rsl_rl.runners import OnPolicyRunner
import torch
import numpy as np
torch.set_printoptions(precision=5, linewidth=200, sci_mode=False)

N          = 100020
state_dim  = 8
action_dim = 5
T          = 60
budget     = 3600.
N_envs     = 1024 
ΔT         = 1.0
device     = 'cpu'
usr_std = 0.1

seed = 0

parameter_static = dict(N          = N,
                        B          = budget,
                        T          = T,                      
                        v_max      = 0.2/14,
                        cost_ti    = 0.0977,
                        cost_ta    = 0.02,
                        cost_v     = 0.07,
                        cost_poc_0 = 0.000369,
                        cost_poc_1 = 0.001057,                      
                        psr        = 0.7,
                        
                        prs        = 1/7)

Random_mean = {"beta2" : 0.78735,
               "beta1" : 0.15747,
               "pei"   : 0.10714,
               "per"   : 0.04545,
               "alpha" : 0.6,
               "pid"   : 1.10/1000,
               "pid_plus" : 0.1221/1000,
               "pir"   :  1/8}

space = {
    key: [value, value * usr_std]
    for key, value in Random_mean.items()
}


Init_state = torch.tensor([[100000 / N, 20 / N, 0, 0, 0, 0, 1, 0]], 
                         dtype=torch.float32).to(torch.device(device))

In [2]:
class myVecEnv(VecEnv):
    def __init__(self, interval, n_envs = N_envs):
        self.B = parameter_static['B']  # total budget
        self.N = parameter_static['N']
        self.device = device
        self.init_state = Init_state
        self.states = self.init_state.repeat(n_envs, 1)
        self.num_envs = n_envs
        self.num_obs = state_dim
        self.num_privileged_obs = None
        self.num_actions = action_dim
        self.max_episode_length = parameter_static['T']
        
        torch.manual_seed(seed)
        self.parameter_random = {
            key: torch.empty(self.num_envs).normal_(
                interval[key][0],
                interval[key][1]
            ).to(torch.device(device))
            for key, value in Random_mean.items()
        }

        # optimization flags for pytorch JIT
        torch._C._jit_set_profiling_mode(False)
        torch._C._jit_set_profiling_executor(False)
        
        # allocate buffers
        self.privileged_obs_buf = None
        self.obs_buf = torch.zeros(self.num_envs, self.num_obs, device=self.device, dtype=torch.float)
        self.rew_buf = torch.zeros(self.num_envs, device=self.device, dtype=torch.float)
        self.reset_buf = torch.ones(self.num_envs, device=self.device, dtype=torch.long)
        self.episode_length_buf = torch.zeros(self.num_envs, device=self.device, dtype=torch.long)
        
        self.actions = torch.zeros(self.num_envs, self.num_actions, dtype=torch.float, device=self.device, requires_grad=False)
        self.episode_sums = torch.zeros(self.num_envs, dtype=torch.float, device=self.device, requires_grad=False)
        self.episode_acts = torch.zeros(self.num_envs, dtype=torch.float, device=self.device, requires_grad=False)
        self.extras = {}
    
    def para_update(self, para):
        self.parameter_random = para
        
    def step(self, actions):
        # actions clip
        soft_bound = [-0.5, 1.5]
        out_of_limits = -(actions - soft_bound[0]).clip(max=0.)
        out_of_limits += (actions - soft_bound[1]).clip(min=0.)
        scale_action = -1
        reward_action = scale_action * torch.sum(out_of_limits, dim=1).clip(max=1.).to(self.device)
        self.actions = torch.clip(actions, 0., 1.).to(self.device)

        # update actions & states
        b, p = self.states[:, -2] * self.B, self.states[:, -1]
        inv_c = self.cost_function(self.actions)
        b -= inv_c
        b = torch.clip(b, 0., self.B).to(self.device)
        mask = (b > 0).float().unsqueeze(-1)
        self.act = self.actions * mask
        inv_c = self.cost_function(self.act)
        next_state, reward_original = self.transition(self.act)
        self.extras['reward_original'] = reward_original
        self.extras['inv_cost'] = inv_c
        self.episode_length_buf += 1
        p = self.episode_length_buf / self.max_episode_length
        self.states = torch.cat([next_state / self.N, (b / self.B).unsqueeze(-1), p.unsqueeze(-1)], dim=-1)
        
        # check termination
        self.reset_buf = self.episode_length_buf >= self.max_episode_length
        
        # compute reward
        r_mean = -6584
        r_std  =  7347
        reward = (reward_original - inv_c - r_mean) / r_std
        self.rew_buf       = reward + reward_action
        self.episode_sums += reward + reward_action
        self.episode_acts += reward_action
        
        # compute resets
        env_ids = self.reset_buf.nonzero(as_tuple=False).flatten()
        self.reset_idx(env_ids)
        
        # compute observations
        return self.get_observations(), self.states, self.rew_buf, self.reset_buf, self.extras

    def reset_idx(self, env_ids):
        """Reset selected robots"""
        if len(env_ids) == 0:
            return
        
        assert len(env_ids) == self.num_envs
        # reset robot states
        self.states[env_ids] = self.init_state
        # reset buffers
        self.episode_length_buf[env_ids] = 0
        self.reset_buf[env_ids] = 1
        # fill extras
        self.extras["episode"] = {}
        self.extras["episode"]['rew_sum'] = torch.mean(self.episode_sums[env_ids]) / (self.max_episode_length * ΔT)
        self.extras["episode"]['rew_act'] = torch.mean(self.episode_acts[env_ids]) / (self.max_episode_length * ΔT)
        self.episode_sums[env_ids] = 0.
        self.episode_acts[env_ids] = 0.
    
    def reset(self):
        """ Reset all robots"""
        self.reset_idx(torch.arange(self.num_envs, device=self.device))
        return {}, {}
    
    def get_observations(self):
        o_mean = torch.tensor([[0.58712867, 0.06804276, 0.00308146, 0.25684301, 0.00261079, 0.08229331, 0.26572794, 0.51      ]], dtype=torch.float32).to(torch.device(self.device))
        o_std  = torch.tensor([[0.41484398, 0.07883003, 0.00389494, 0.292568  , 0.00394907, 0.06906768, 0.33943075, 0.28861739]], dtype=torch.float32).to(torch.device(self.device))
        self.obs_buf = (self.states - o_mean) / o_std
        return self.obs_buf
    
    def get_privileged_observations(self):
        return self.privileged_obs_buf

    def cost_function(self, action):
        states = self.states[:,:-2] * self.N
        st, et, at, it, dt, rt = states[:, 0], states[:, 1], states[:, 2], states[:, 3], states[:, 4], states[:, 5]
        u_sv, u_it, u_aq, u_iq, w = action[:, 0], action[:, 1], action[:, 2], action[:, 3], action[:, 4]

        cost_it  = it * u_it * parameter_static['cost_ti']
        cost_v   = st * u_sv * parameter_static['cost_v'] * parameter_static['v_max']
        cost_q   = (u_aq * at + u_iq * it) * parameter_static['cost_ta']
        per_poc  = ((st + et + at + rt) / self.N).pow(2) * parameter_static['cost_poc_0'] + parameter_static['cost_poc_1']
        cost_poc = (st + et + rt + at) * w * per_poc
        return cost_poc + cost_it + cost_v + cost_q

    def transition(self, action):
        states = self.states[:,:-2] * self.N
        st, et, at, it, dt, rt = states[:, 0], states[:, 1], states[:, 2], states[:, 3], states[:, 4], states[:, 5]
        u_sv, u_it, u_aq, u_iq, w = action[:, 0], action[:, 1], action[:, 2], action[:, 3], action[:, 4]
        ht = st + et + at + it + rt

        st_out   = (1 - u_sv * parameter_static['v_max']) * st * (it * (1 - u_iq) * self.parameter_random['beta1'] + (et + at * (1 - u_aq)) * self.parameter_random['beta2']) / ht
        st_out_e = st_out * self.parameter_random['alpha']
        st_out_i = st_out * (1 - self.parameter_random['alpha'])
        st_out_r = u_sv * parameter_static['v_max'] * st * parameter_static['psr']
        et_out_r = et * self.parameter_random['per']
        et_out_i = et * self.parameter_random['pei']
        et_out_a = et * w * (1 - self.parameter_random['per'] - self.parameter_random['pei'])
        at_out_i = at * self.parameter_random['pei']
        at_out_r = at * self.parameter_random['per']
        it_out_r = it * self.parameter_random['pir'] * u_it
        it_out_d = it * self.parameter_random['pid'] * (1 - u_it) + it * u_it * self.parameter_random['pid_plus']
        rt_out_s = rt * parameter_static['prs']
        
        stn = st - st_out_e - st_out_i - st_out_r + rt_out_s
        etn = et - et_out_r - et_out_i - et_out_a + st_out_e
        atn = at - at_out_r - at_out_i + et_out_a
        itn = it - it_out_r - it_out_d + st_out_i + et_out_i + at_out_i
        dtn = dt + it_out_d
        rtn = rt + et_out_r + at_out_r + it_out_r + st_out_r - rt_out_s

        reward = -2.6 * st_out - 100 * it_out_d

        next_state = torch.stack([stn, etn, atn, itn, dtn, rtn], dim=-1)
        return next_state, reward

In [3]:
def para_sampling(rand_para, intvl, num_sample=N_envs):
    parameter_random = {
        key: torch.empty(num_sample).normal_(
        intvl[key][0],
        intvl[key][1]
        ).to(torch.device(device))
        for key, value in rand_para.items()
    }
    return parameter_random
    
def update_space(env_sample, final_indices, space, unc):
    para_value = {}
    for key, tensor_values in env_sample.items():
        selected_values = torch.mean(tensor_values[final_indices]) 
        update_values = space[key][0] + 0.5 * (selected_values - space[key][0])
        space[key] = [update_values , update_values  * usr_std]
        para_value[key] = selected_values
    return space, para_value

def cal_err(para_value, env_para):
    gap = []
    for key, value in env_para.items():
        val = value.item()  # 获取 tensor 内部的数值
        gap_temp = abs(para_value[key] - env_para[key])/(env_para[key])
        gap.append(gap_temp)
    return sum(gap)/len(gap) # 所有值都在范围内，返回 0
    
def state_estimator(env_sample, action_sim, state_sim, state_prior, para): 
    env_sample.init_state = state_prior
    env_sample.reset()
    env_sample.para_update(para)
    obs_sample, states_sample, _, _, _ = env_sample.step(action_sim.detach())  
    mse = torch.mean((state_sim[:,2:4] - states_sample[:,2:4]) ** 2, dim=1) 
    mask = para_mask(env_sample.parameter_random)
    top_indices = torch.argsort(mse)[:int(len(mse) * 5e-2)] 
    final_indices = torch.tensor(list(set(mask.tolist()) & set(top_indices.tolist())), device=device)

    state_est = torch.mean(states_sample[final_indices,:], dim=0)
    state_est[-1] = state_prior[-1] + torch.tensor(1/T)  
    return state_est, final_indices

In [4]:
para_err = 0.25
torch.manual_seed(seed)
para_true = []
for t in range(T):
    para_temp = {
        key: (Random_mean[key] - Random_mean[key] * torch.sin(torch.tensor(t/(3.1415926 * 3))) * para_err + 
              torch.empty(1).normal_(0, Random_mean[key] * 0.01)).to(torch.device(device))
        for key, value in Random_mean.items()
        }
    para_true.append(para_temp)

err = []
err_truth = []
env = myVecEnv(interval = space, n_envs=1)
env.para_update(para_true[0])
env_sample = myVecEnv(space)
state_est = Init_state[0]
for t in range(T-1):
    actions = torch.rand([1,5]) * 1e-2
    obs, states, rews, dones, infos = env.step(actions.detach())
    para_sam = para_sampling(env.parameter_random, space)
    state_est, top = state_estimator(env_sample, actions, states, state_est, para_sam)
    space, para_v = update_space(env_sample.parameter_random, top, space, usr_std) 
    err_temp = cal_err(para_v, env.parameter_random)
    ett = cal_err(Random_mean, env.parameter_random)
    if t < T - 1: env.para_update(para_true[t + 1])
    err.append(err_temp) 
    err_truth.append(ett)
env.reset()
print(torch.mean(torch.tensor(err)))
#print(torch.mean(torch.tensor(err_truth)))

tensor(0.23359)


In [5]:
max(err)

tensor([0.41178])

In [6]:
min(err)

tensor([0.01294])

In [7]:
std_err = []
for i in range(len(err)):
    std_err.append(np.array(err[i]))
np.std(np.array(std_err))

0.10515631